# Ekstraksi Fitur Data kualitas udara di wilayah Dukun, Gresik


## Ekstraksi 68 Fitur TSFEL untuk Polutan NO2

Tahap ini mengekstraksi 68 fitur numerik dari data time series konsentrasi NO2 harian (kolom NO2_imputed) menggunakan pustaka TSFEL. Setiap fitur merepresentasikan satu karakteristik statistik, temporal, spektral, atau kompleksitas dari sinyal, sehingga deret waktu 366 hari diringkas jadi satu baris fitur per kecamatan. Fungsi diambil langsung dari tsfel.feature_extraction.features (bukan lewat extractor tingkat tinggi) agar daftar fitur persis 68 sesuai permintaan, dengan fs=1 karena data harian.

68 fitur dikelompokkan jadi domain2:

- Statistik — sebaran nilai NO2 secara keseluruhan: calc_mean, calc_median, calc_std, calc_var, calc_max, calc_min, skewness, kurtosis, interq_range, mean_abs_deviation, median_abs_deviation, ecdf, ecdf_percentile, ecdf_percentile_count, ecdf_slope, hist_mode, pk_pk_distance, rms
- Temporal — pola perubahan hari ke hari & kompleksitas/fraktal sinyal: autocorr, distance, slope, entropy, zero_cross, mean_diff, median_diff, mean_abs_diff, median_abs_diff, sum_abs_diff, positive_turning, negative_turning, neighbourhood_peaks, calc_centroid, auc, abs_energy, mse, dfa, hurst_exponent, higuchi_fractal_dimension, petrosian_fractal_dimension, maximum_fractal_length, lempel_ziv
- Spektral — pola frekuensi/musiman hasil transformasi Fourier: spectral_centroid, spectral_spread, spectral_skewness, spectral_kurtosis, spectral_entropy, spectral_slope, spectral_decrease, spectral_distance, spectral_variation, spectral_positive_turning, spectral_roll_off, spectral_roll_on, max_frequency, median_frequency, fundamental_frequency, max_power_spectrum, power_bandwidth, average_power, human_range_energy, mfcc, lpcc, spectrogram_mean_coeff

In [1]:
!pip install tsfel

In [2]:
import pandas as pd
import numpy as np
import inspect
import tsfel.feature_extraction.features as tsfel_features

# ---------- 1. Muat dan bersihkan data ----------
df = pd.read_csv('NO2_Dukun_Cleaned.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

target_pollutant = 'NO2'

# --- FIX: paksa kolom target jadi numerik, nilai yang gagal dikonversi -> NaN ---
df[target_pollutant] = pd.to_numeric(df[target_pollutant], errors='coerce')

n_missing_before = df[target_pollutant].isna().sum()
print(f"Jumlah nilai non-numerik/kosong yang dikonversi jadi NaN: {n_missing_before}")

Q1 = df[target_pollutant].quantile(0.25)
Q3 = df[target_pollutant].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
df.loc[(df[target_pollutant] < lower_bound) | (df[target_pollutant] > upper_bound), target_pollutant] = np.nan

df_clean = df.set_index('date').interpolate(method='time').ffill().bfill()

fs = 1
signal_1d = df_clean[target_pollutant].astype(float).values

# ---------- 2. Daftar PERSIS fitur yang diminta ----------
FEATURE_LIST = """abs_energy auc autocorr average_power calc_centroid calc_max calc_mean
calc_median calc_min calc_std calc_var dfa distance ecdf ecdf_percentile ecdf_percentile_count
ecdf_slope entropy fundamental_frequency higuchi_fractal_dimension hist_mode human_range_energy
hurst_exponent interq_range kurtosis lempel_ziv lpcc max_frequency max_power_spectrum
maximum_fractal_length mean_abs_deviation mean_abs_diff mean_diff median_abs_deviation
median_abs_diff median_diff median_frequency mfcc mse negative_turning neighbourhood_peaks
petrosian_fractal_dimension pk_pk_distance positive_turning power_bandwidth rms skewness slope
spectral_centroid spectral_decrease spectral_distance spectral_entropy spectral_kurtosis
spectral_positive_turning spectral_roll_off spectral_roll_on spectral_skewness spectral_slope
spectral_spread spectral_variation spectrogram_mean_coeff sum_abs_diff wavelet_abs_mean
wavelet_energy wavelet_entropy wavelet_std wavelet_var zero_cross""".split()

print("Jumlah fitur yang diminta:", len(FEATURE_LIST))


def to_scalar(result):
    if isinstance(result, dict) and "values" in result:
        result = result["values"]
    if isinstance(result, (list, tuple, np.ndarray)):
        arr = np.asarray(result, dtype=float)
        return float(np.nanmean(arr))
    return float(result)


def extract_one(fn_name, signal, fs):
    fn = getattr(tsfel_features, fn_name)
    params = inspect.signature(fn).parameters
    if "fs" in params:
        result = fn(signal, fs)
    else:
        result = fn(signal)
    return to_scalar(result)


row = {}
for fn_name in FEATURE_LIST:
    row[fn_name] = extract_one(fn_name, signal_1d, fs)

extracted_features_final = pd.DataFrame([row])

print(f"Berhasil! Jumlah fitur yang dihasilkan untuk {target_pollutant}: {extracted_features_final.shape[1]}")
extracted_features_final.to_csv(f'{target_pollutant}_Dukun_TSFEL.csv', index=False)

Jumlah nilai non-numerik/kosong yang dikonversi jadi NaN: 159
Jumlah fitur yang diminta: 68


Berhasil! Jumlah fitur yang dihasilkan untuk NO2: 68


In [3]:
import pandas as pd

df_fitur = pd.read_csv("NO2_Dukun_TSFEL.csv")
pd.set_option('display.max_columns', None)
display(df_fitur)

,abs_energy,auc,autocorr,average_power,calc_centroid,calc_max,calc_mean,calc_median,calc_min,calc_std,calc_var,dfa,distance,ecdf,ecdf_percentile,ecdf_percentile_count,ecdf_slope,entropy,fundamental_frequency,higuchi_fractal_dimension,hist_mode,human_range_energy,hurst_exponent,interq_range,kurtosis,lempel_ziv,lpcc,max_frequency,max_power_spectrum,maximum_fractal_length,mean_abs_deviation,mean_abs_diff,mean_diff,median_abs_deviation,median_abs_diff,median_diff,median_frequency,mfcc,mse,negative_turning,neighbourhood_peaks,petrosian_fractal_dimension,pk_pk_distance,positive_turning,power_bandwidth,rms,skewness,slope,spectral_centroid,spectral_decrease,spectral_distance,spectral_entropy,spectral_kurtosis,spectral_positive_turning,spectral_roll_off,spectral_roll_on,spectral_skewness,spectral_slope,spectral_spread,spectral_variation,spectrogram_mean_coeff,sum_abs_diff,wavelet_abs_mean,wavelet_energy,wavelet_entropy,wavelet_std,wavelet_var,zero_cross
0,3.782973e-07,0.010983,3.0,1.036431e-09,203.472414,0.000071,0.00003,0.000028,0.000007,0.000011,1.293063e-10,0.869955,365.0,0.015027,0.00003,182.5,29976.74047,0.999358,0.002732,1.920019,0.00003,0.0,0.783378,0.000014,0.806169,0.18306,0.6677,0.42623,67.045594,-2.519529,0.000009,0.000006,-3.717708e-08,0.000007,0.000003,-2.173505e-07,0.087432,16.896036,1.036666,57.0,15.0,1.020639,0.000063,58.0,0.29235,0.000032,0.954705,2.748443e-08,0.131274,-2.098373,-1.763756,0.73751,2.709172,61.0,0.42623,0.0,0.927511,-0.030637,0.142728,0.332346,1.727637e-10,0.002232,0.000002,0.000017,2.161638,0.000017,2.963959e-10,0.0


## Simpan Hasil Ekstraksi ke Database Aiven

Contoh ini menggunakan **PostgreSQL** (jenis service Aiven yang paling umum dipakai).

Sebelum menyambungkan koneksi melalui aplikasi apa pun, kita membutuhkan informasi kredensial server.
1. Akses *dashboard* atau console **Aiven**, lalu arahkan ke proyek yang dimiliki.
2. Buka tab **Overview** pada layanan (*service*) PostgreSQL yang sedang beroperasi (`pg-157c2e9`).
3. Pada bagian **Connection information**, catat parameter-parameter berikut ini:
   * **Host:** `pg-157c2e9-project-95fb.l.aivencloud.com`
   * **Port:** `13459`
   * **User:** `avnadmin`
   * **Password:** `AVNS_Khi-ZHOCTI3NhlU_Dui`
   * **SSL mode:** `require`
   * **Database:** `udara_dukun_tsfel`

![Aiven PostgreSQL Console](../img/aiven2.png)


### Data di Database


![Aiven PostgreSQL Console](../img/dbdukun.png)